# Arithmetic Convention

linopy enforces strict defaults for coordinate alignment so that mismatches never silently produce wrong results.

Two rules apply to **all** arithmetic operations involving linopy objects (`+`, `-`, `*`, `/`):

**Rule 1 — Exact label matching on shared dimensions**

When two operands share a dimension, their coordinate labels on that dimension must match exactly (`join="exact"`). A `ValueError` is raised on mismatch.

**Rule 2 — Constants cannot introduce new dimensions**

When combining an expression or variable with a *constant* (`DataArray`, numpy, pandas), the constant's dimensions must be a subset of the expression's dimensions. A constant cannot introduce dimensions the expression does not have — that would silently duplicate variables.

Expression + Expression broadcasting over non-shared dimensions is freely allowed.

Inspired by [pyoframe](https://github.com/Bravos-Power/pyoframe).

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

import linopy

## Setup

In [ ]:
m = linopy.Model()

time = pd.RangeIndex(5, name="time")
techs = pd.Index(["solar", "wind", "gas"], name="tech")
scenarios = pd.Index(["low", "high"], name="scenario")

x = m.add_variables(lower=0, coords=[time], name="x")
y = m.add_variables(lower=0, coords=[time], name="y")
gen = m.add_variables(lower=0, coords=[time, techs], name="gen")
risk = m.add_variables(lower=0, coords=[techs, scenarios], name="risk")

## What works by default

In [ ]:
# Same coords — just works
x + y

In [ ]:
# Constant with matching coords
factor = xr.DataArray([2, 3, 4, 5, 6], dims=["time"], coords={"time": time})
x * factor

In [ ]:
# Constant with fewer dims — broadcasts freely
cost = xr.DataArray([1.0, 0.5, 3.0], dims=["tech"], coords={"tech": techs})
gen * cost  # cost broadcasts over time

In [ ]:
# Expression + Expression with non-shared dims — broadcasts freely
gen + risk  # (time, tech) + (tech, scenario) → (time, tech, scenario)

In [ ]:
# Scalar — always fine
x + 5

In [ ]:
# Constraints — RHS with fewer dims broadcasts naturally
capacity = xr.DataArray([100, 80, 50], dims=["tech"], coords={"tech": techs})
m.add_constraints(gen <= capacity, name="cap")  # capacity broadcasts over time

## What raises an error

In [ ]:
# Mismatched coordinates on shared dimension
y_short = m.add_variables(
    lower=0, coords=[pd.RangeIndex(3, name="time")], name="y_short"
)

try:
    x + y_short  # time coords don't match
except ValueError as e:
    print("ValueError:", e)

In [ ]:
# Constant introduces new dimensions
profile = xr.DataArray(
    np.ones((3, 5)), dims=["tech", "time"], coords={"tech": techs, "time": time}
)
try:
    (
        x + profile
    )  # would duplicate x[t] across techs. Reduce using mean, max or sth similar
except ValueError as e:
    print("ValueError:", e)

In [ ]:
# Multiplication with mismatched coordinates
partial = xr.DataArray([10, 20, 30], dims=["time"], coords={"time": [0, 1, 2]})
try:
    x * partial  # time coords [0..4] vs [0,1,2]
except ValueError as e:
    print("ValueError:", e)

In [ ]:
# Constraint RHS with mismatched coordinates
partial_rhs = xr.DataArray([10, 20, 30], dims=["time"], coords={"time": [0, 1, 2]})

try:
    x <= partial_rhs
except ValueError as e:
    print("ValueError:", e)

## Escape hatches

When coordinates don't match, linopy provides several ways to state your intent explicitly.

### 1. `.sel()` — Subset before operating

The cleanest way to restrict to matching coordinates. No need for an inner join — explicitly select what you want.

In [ ]:
x.sel(time=[0, 1, 2]) + y_short  # select matching coords first

### 2. `| 0` — Inline outer join with fill

When one operand covers a subset of coordinates, use the `|` operator to declare a fill value. This creates a lightweight `FillWrapper` that is consumed immediately — it never mutates or persists.

The `|` operator only works on linopy types (`Variable`, `LinearExpression`, `QuadraticExpression`). For external types, use `.reindex()` before operating.

In [ ]:
x + (y_short | 0)  # fill missing time coords of y_short with 0

### 3. Named methods with `join=`

All arithmetic operations have named-method equivalents that accept a `join` parameter:

| `join` | Coordinates kept | Fill |
|--------|-----------------|------|
| `"exact"` | Must match | `ValueError` if different |
| `"inner"` | Intersection | — |
| `"outer"` | Union | Zero (arithmetic) / NaN (constraints) |
| `"left"` | Left operand's | Zero / NaN for missing right |
| `"right"` | Right operand's | Zero for missing left |
| `"override"` | Left operand's | Positional alignment |

In [ ]:
m2 = linopy.Model()

i_a = pd.Index([0, 1, 2], name="i")
i_b = pd.Index([1, 2, 3], name="i")

a = m2.add_variables(coords=[i_a], name="a")
b = m2.add_variables(coords=[i_b], name="b")

print("inner:", list(a.add(b, join="inner").coords["i"].values))  # [1, 2]
print("outer:", list(a.add(b, join="outer").coords["i"].values))  # [0, 1, 2, 3]
print("left: ", list(a.add(b, join="left").coords["i"].values))  # [0, 1, 2]
print("right:", list(a.add(b, join="right").coords["i"].values))  # [1, 2, 3]

### 4. `linopy.align()` — Explicit pre-alignment

For complex multi-operand alignment:

In [ ]:
a_aligned, b_aligned = linopy.align(a, b, join="outer", fill_value=0)
a_aligned + b_aligned

## Positional alignment

When two arrays have the same shape but different coordinate labels, use `.assign_coords()` to relabel one operand so coordinates match explicitly:

In [ ]:
c = m2.add_variables(coords=[["x", "y", "z"]], name="c")
d = m2.add_variables(coords=[["p", "q", "r"]], name="d")

# Relabel d's coordinates to match c, then add
c + d.assign_coords(dim_0=c.coords["dim_0"])

In [ ]:
# Or use join="override" for positional matching
c.add(d, join="override")

## Working with pandas

Under the strict convention, pandas objects must have **named indices** to avoid dimension name mismatches. A `pd.Series` without a named index becomes `dim_0` and will fail the exact join against a named variable dimension.

```python
# Bad — index name is None, becomes "dim_0"
cost = pd.Series([10, 20], index=["wind", "solar"])

# Good — explicit dimension name
cost = pd.Series([10, 20], index=pd.Index(["wind", "solar"], name="tech"))
```

Consider using `force_dim_names=True` on the model to catch unnamed dimension issues at variable creation time.

## Summary

| Situation | Behavior | How to handle |
|---|---|---|
| Shared dims, matching coords | ✓ Proceeds | `x + y` |
| Non-shared dims, expr + expr | ✓ Broadcasts | `gen(time,tech) + risk(tech,scenario)` |
| Constant with subset dims | ✓ Broadcasts | `cost(tech) * gen(tech,time)` |
| Constant introduces new dims | ✗ Raises | Restructure, or multiply if meaningful |
| Shared dims, mismatching coords | ✗ Raises | `.sel()` or `x + (y \| 0)` |
| Pandas without named index | ✗ Raises on dim mismatch | Name the index |